# Imports Libraries, Dependencies & Datasets

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
import re
import string

from sklearn.metrics._plot.confusion_matrix import confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
data_fake = pd.read_csv("Fake.csv");
data_true = pd.read_csv("True.csv");

# Analisi dei Dati

In [ ]:
# Aggiungiamo la Clonna Class ai Dati
data_fake["label"] = 0;
data_true["label"] = 1;

In [ ]:
data_fake.head()

In [ ]:
data_true.head()

In [ ]:
data_fake.shape, data_true.shape

# Trattamento dei Dati

In [ ]:
# Concateniamo i DataSet data_fake e data_true in un nuovo dataset data_merge
data_merge = pd.concat([data_fake, data_true], axis=0);

In [ ]:
data_merge.head(10)

In [ ]:
data_merge.tail(10)

In [ ]:
# Eliminamo le Colonne che non ci servono
data_merge.columns

In [ ]:
data = data_merge.drop(["title", "subject", "date"], axis=1)

In [ ]:
data.columns

In [ ]:
# Mischiamo le righe del DataSet (Random Shuffling)
data = data.sample(frac=1);
data.head()

In [ ]:
# Resettiamo l'index
data.reset_index(inplace=True)
data.head()

In [ ]:
# Droppiamo la Colonna Index
data = data.drop(["index"], axis=1)
data.head()

# Processiamo il Testo


In [ ]:
# Funzione di ottimizzazione del testo
# Rimuoviamo e sostituiamo tutte le parti del testo che non hanno
# un significato
def wordopt(text):
  # Convert to lower-case
  text = text.lower()

  # Remove URLs
  text = re.sub(r"https?://\S+|www\.\S+", " ", text)

  # Remove HTML tags
  text = re.sub(r"<.*?>+", " ", text)

  # Remove Puntuaction
  text = re.sub(r"[^\w\s]", " ", text)

  # Remove Digits
  text = re.sub(r"\d", " ", text)

  # Rimozione Rumore
  text = re.sub(r"reuters", " ", text)

  # Remove Newline Characters
  text = re.sub("\n", " ", text)

  return text

In [ ]:
data["text"] = data["text"].apply(wordopt)
data.head()

In [ ]:
# Dividiamo il DataSet in train e test
x = data["text"]
y = data["label"]
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3)

In [ ]:
x_train.shape, y_train.shape

In [ ]:
x_test.shape, y_test.shape

# Vettorizzazione del Testo
Passiamo alla fase di Feature Extraction. Per adesso i dati sono in una forma Testuale. Il nostro obiettivo e' quello di riportarli in un formato numerico, migliore per l'addestramento del modello di Machine Learning.

Trasformeremo ogni parola in un vettore numerico n-dimensionale il quale conterra' le proprieta' di quest'ultima.



In [ ]:
vectorization = TfidfVectorizer()
xv_train = vectorization.fit_transform(x_train)
xv_test = vectorization.transform(x_test)

### **Modelli di Machine Learning**

# **Linear Regression**

In [ ]:
LR = LogisticRegression()
LR.fit(xv_train, y_train)

In [ ]:
pred_lr = LR.predict(xv_test)
LR.score(xv_test, y_test)

In [ ]:
print(classification_report(y_test, pred_lr))

In [ ]:
cm_lr = confusion_matrix(y_test, pred_lr)
cm_lr = pd.DataFrame(cm_lr , index = ['TRUE','FAKE'] , columns = ['TRUE','FAKE'])
plt.figure(figsize = (6,4))
sns.heatmap(cm_lr,cmap= "Blues", linecolor = 'black' , linewidth = 1 , annot = True, fmt='' , xticklabels = ['TRUE','FAKE'] , yticklabels = ['TRUE','FAKE'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [ ]:
i_falsePositive_lr = np.argwhere((y_test == 0) & pred_lr).flatten()
i_falsePositive_lr.shape

In [ ]:
i_falseNegative_lr = np.argwhere(y_test & (pred_lr == 0)).flatten()
i_falseNegative_lr.shape

# **Decision Tree**

In [ ]:
DT = DecisionTreeClassifier()
DT.fit(xv_train, y_train);
DT.get_depth()

In [ ]:
pred_dt = DT.predict(xv_test)
DT.score(xv_test, y_test)

In [ ]:
print(classification_report(y_test, pred_dt))

In [ ]:
cm_dt = confusion_matrix(y_test, pred_dt)
cm_dt = pd.DataFrame(cm_dt , index = ['TRUE','FAKE'] , columns = ['TRUE','FAKE'])
plt.figure(figsize = (6,4))
sns.heatmap(cm_dt, cmap= "Blues", linecolor = 'black' , linewidth = 1 , annot = True, fmt='' , xticklabels = ['TRUE','FAKE'] , yticklabels = ['TRUE','FAKE'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [ ]:
i_falsePositive_dt = np.argwhere((y_test == 0) & pred_dt).flatten()
i_falsePositive_dt.shape

In [ ]:
i_falseNegative_dt = np.argwhere(y_test & (pred_dt == 0)).flatten()
i_falseNegative_dt.shape

In [ ]:

# Recuperiamo i nomi reali delle parole
parole_chiave = vectorization.get_feature_names_out()

# Estrazione dell'importanza delle feature
importanze = DT.feature_importances_

# Creiamo un DataFrame per ordinare e filtrare solo le parole usate dall'albero (importanza > 0)
df_importanze = pd.DataFrame({"Parola": parole_chiave, "Importanza": importanze})
df_importanze = df_importanze[df_importanze["Importanza"] > 0].sort_values(by="Importanza", ascending=True).tail(8)

# Plot del grafico a barre delle Parole Chiave decisionali
plt.figure(figsize=(10, 6), dpi=300)
colori = plt.cm.viridis(np.linspace(0.4, 0.8, len(df_importanze)))

plt.barh(df_importanze["Parola"], df_importanze["Importanza"], color=colori, edgecolor="black", height=0.6)

# Estetica del grafico
plt.xlabel("Livello di Importanza nel processo decisionale", fontsize=11, fontweight="bold")
plt.ylabel("Parole Chiave", fontsize=11, fontweight="bold")
plt.title("Le parole Chiave dell'Albero Decisionale", fontsize=13, fontweight="bold", pad=15)
plt.grid(axis="x", linestyle="--", alpha=0.7)

plt.tight_layout()
plt.show()

# **Random Forest**

In [ ]:
vectorization_rf = TfidfVectorizer(max_features=2000, min_df=5)
xv_train_rf = vectorization_rf.fit_transform(x_train)
xv_test_rf = vectorization_rf.transform(x_test)

In [ ]:
RF = RandomForestClassifier(max_depth=30, n_estimators=300);
RF.fit(xv_train_rf, y_train)

In [ ]:
pred_rf = RF.predict(xv_test_rf)
RF.score(xv_test_rf, y_test)

In [ ]:
print(classification_report(y_test, pred_rf))

In [ ]:
cm = confusion_matrix(y_test, pred_rf)
cm = pd.DataFrame(cm , index = ['TRUE','FAKE'] , columns = ['TRUE','FAKE'])
plt.figure(figsize = (6,4))
sns.heatmap(cm,cmap= "Blues", linecolor = 'black' , linewidth = 1 , annot = True, fmt='' , xticklabels = ['TRUE','FAKE'] , yticklabels = ['TRUE','FAKE'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [ ]:
i_falsePositive_rf = np.argwhere((y_test == 0) & pred_rf).flatten()
i_falsePositive_rf.shape

In [ ]:
i_falseNegative_rf = np.argwhere(y_test & (pred_rf == 0)).flatten()
i_falseNegative_rf.shape

In [ ]:
parole_chiave = vectorization_rf.get_feature_names_out()

# Estrazione dell'importanza delle feature
importanze = RF.feature_importances_

# Creiamo un DataFrame per ordinare e filtrare solo le parole usate dall'albero (importanza > 0)
df_importanze = pd.DataFrame({"Parola": parole_chiave, "Importanza": importanze})
df_importanze = df_importanze[df_importanze["Importanza"] > 0].sort_values(by="Importanza", ascending=True).tail(8)

# Plot del grafico a barre delle Parole Chiave decisionali
plt.figure(figsize=(10, 6), dpi=300)
colori = plt.cm.viridis(np.linspace(0.4, 0.8, len(df_importanze)))

plt.barh(df_importanze["Parola"], df_importanze["Importanza"], color=colori, edgecolor="black", height=0.6)

# Estetica del grafico
plt.xlabel("Livello di Importanza nel processo decisionale (Gini)", fontsize=11, fontweight="bold")
plt.ylabel("Parole Chiave", fontsize=11, fontweight="bold")
plt.title("Le parole d'oro dell'Albero Decisionale\n(I termini che discriminano meglio le Fake News)", fontsize=13, fontweight="bold", pad=15)
plt.grid(axis="x", linestyle="--", alpha=0.7)

plt.tight_layout()
plt.show()

# **Testing**

In [ ]:
def histogram(lista_numeri):
  numeri, frequenze = np.unique(lista_numeri, return_counts=True)

  indici_ordinati = np.argsort(numeri)
  numeri_ordinati = numeri[indici_ordinati]
  frequenze_ordinate = frequenze[indici_ordinati]

  asse_x_etichette = [str(n) for n in numeri_ordinati]

  sns.set_theme(style="whitegrid")
  plt.figure(figsize=(50, 5))
  plt.bar(
      asse_x_etichette,
      frequenze_ordinate,
      width=1.0,
      color="#4C72B0",
      edgecolor="black",
      linewidth=0.3,
  )
  plt.title(
      "Indici",
      fontsize=16,
      fontweight="bold",
      pad=15,
  )
  plt.xlabel("Indice", fontsize=13, labelpad=10)
  plt.ylabel("Frequenza (Conteggio)", fontsize=13, labelpad=10)
  plt.xticks(
      ticks=range(0, len(asse_x_etichette), 1),
      labels=asse_x_etichette[::1],
      rotation=45,
      fontsize=10,
  )
  plt.xlim(-0.5, len(asse_x_etichette) - 0.5)
  plt.tight_layout()
  plt.show()


def removeUniqueIndices(lista_numeri):
  valori, conteggi = np.unique(lista_numeri, return_counts=True)
  valori_ripetuti = valori[conteggi > 1]
  maschera = np.isin(lista_numeri, valori_ripetuti)
  return lista_numeri[maschera]


In [ ]:
i_falsePositive = np.concat((i_falsePositive_lr, i_falsePositive_dt, i_falsePositive_rf))
histogram(removeUniqueIndices(i_falsePositive))

In [ ]:
i_falseNegative = np.concat((i_falseNegative_lr, i_falseNegative_dt, i_falseNegative_rf))
histogram(removeUniqueIndices(i_falseNegative))

In [ ]:

parole_chiave_lr = vectorization.get_feature_names_out()
parole_chiave_rf = vectorization_rf.get_feature_names_out()

# Estrazione delle metriche
pesi_lineari = LR.coef_.flatten()
df_lineare = pd.DataFrame({"Parola": parole_chiave_lr, "Peso": pesi_lineari})
df_lineare = df_lineare.reindex(df_lineare.Peso.abs().sort_values(ascending=False).index).head(8)
df_lineare = df_lineare.sort_values(by="Peso")

# Random Forest: l'importanza media delle feature
importanze_rf = RF.feature_importances_
df_rf = pd.DataFrame({"Parola": parole_chiave_rf, "Importanza": importanze_rf})
df_rf = df_rf[df_rf["Importanza"] > 0.005].sort_values(by="Importanza", ascending=True).tail(8)

# Creazione del Grafico Comparativo
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7), dpi=300)

# Grafico 1: Modello Lineare
colori_lineari = ['darkgreen' if x > 0 else 'darkred' for x in df_lineare["Peso"]]
ax1.barh(df_lineare["Parola"], df_lineare["Peso"], color=colori_lineari, edgecolor="black", height=0.6)
ax1.axvline(0, color='black', linestyle='-', linewidth=1.2)
ax1.set_title("Modello Lineare: Pesi delle Parole\n(Rossi = Spingono verso FAKE | Verdi = Verso VERA)", fontsize=12, fontweight="bold")
ax1.set_xlabel("Valore del Coefficiente (Impatto e Direzione)", fontsize=10)
ax1.grid(axis="x", linestyle="--", alpha=0.5)


# Grafico 2: Random Forest
colori_rf = plt.cm.plasma(np.linspace(0.4, 0.8, len(df_rf)))
ax2.barh(df_rf["Parola"], df_rf["Importanza"], color=colori_rf, edgecolor="black", height=0.6)
ax2.set_title("Random Forest: Importanza", fontsize=12, fontweight="bold")
ax2.set_xlabel("Grado di Importanza Medio", fontsize=10)
ax2.grid(axis="x", linestyle="--", alpha=0.5)

plt.suptitle("Confronto: Modello Lineare vs Random Forest", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


